In [ ]:
using BSON, Dates, DelimitedFiles, Downloads, CUDA, cuDNN, Flux, Printf, Plots, JLD2, Distributions,LinearAlgebra
using Flux.Zygote
include("neural.jl")
include("utils.jl")
include("radialc2.jl");

In [ ]:
BSON.@load "model_meta.bson" model #load model

In [ ]:

function get_c1_neural_rad(model,scale_windows,scale_dict)
    window_bins = 401 #input bins for density
    model = model |> gpu
    function (ρ, ϕ, xs)
        l = length(ρ)
        ρ_windows = generate_windows(ρ; window_bins)
        ϕ_func = generate_phi(ϕ,ρ)|> gpu
        r = [-reverse(xs);  xs] #extendet to -r
        ρ_windows = scale_windows(ρ_windows,r,scale_dict) |> gpu #preprocessing density
        
        rint = R_meta.(r)|> gpu #preprocessing position
        input = vcat(ρ_windows, ϕ_func,rint')|> gpu  
        c1 = model(input) |> cpu |> vec #evaluate model
        c11 = c1[Integer(l/2+1):end]
        c12 = reverse(c1[1:Integer(l/2)]) #mirrored output
        (c11+c12)/2 #symmetrized version, Eq. (19)
  
    end
end


function minimize(L::Number, μ::Number, T::Number, ϕ, Vext::Function, c1neural::Function, model; scalewindows=nothing, α::Number=0.03, maxiter::Int=10000, dx::Number=0.01, floattype::Type=Float32, tol::Number=max(eps(floattype(1e3)), 1e-8))
    L, μ, T = floattype.((L, μ, T))
    ϕ = vec(ϕ);
    ϕ = floattype.(ϕ)
    xs = collect(floattype, dx/2:dx:L)
    Vext = Vext.(xs) #evaluate external potential
    infiniteVext = isinf.([reverse(Vext);Vext])  
    ρ, ρEL = zero([xs;xs]), zero([xs;xs]) #preallocate density
    fill!(ρ, 0.5) 
    βVext = Vext ./ T #thermal scaling
    βμ = μ / T
    βϕ = ϕ ./ T
    rdict = construct_scale_dict(xs);  
    if scalewindows != nothing
        c1_func = c1neural(model,scalewindows,rdict)
    else
        c1_func = c1neural(model)
    end
    i = 0
    while true #Picard iteration
        c1 = c1_func(ρ,βϕ,xs)
        ρEL .= exp.(βμ .- [reverse(βVext);βVext] .+ [reverse(c1);c1])  # Evaluate the RHS of the Euler-Lagrange equation
        ρ .= (1 - α) .* ρ .+ α .* ρEL 
        ρ[infiniteVext] .= 0  
        clamp!(ρ, 0, Inf)  
        Δρmax = maximum(abs.(ρ - ρEL)[.!infiniteVext])  
        i += 1
        
        if Δρmax < tol
            println("Converged (step: $(i), ‖Δρ‖ = $(Δρmax) < $(tol) = tolerance)")
            break 
        end
        if !isfinite(Δρmax) || i >= maxiter
            println("Did not converge (step: $(i) of $(maxiter), ‖Δρ‖: $(Δρmax), tolerance: $(tol))")
            return xs, ρ[Integer(L/dx+1):end]  
        end
    end
    xs, ρ[Integer(L/dx+1):end]
end

In [ ]:
#possible pair potentials
function ramp(x)
    if x < 0.5
        return 3-x
    elseif 0.5 <= x < 1
        return 2-0.5*x
    else
        return 0
    end
end

function soft(x)
    a = 1.5
    return a*exp(-2*x^6) - a*exp(-2*(1.5)^6 )
end


function gauss(x)
    a = 4
    return a*exp(-2*(1.2*x)^2) - a*exp(-2*(1.5*1.2)^2)
end

function sqs(x)
    if x < 1
        return Inf
    elseif 1 <= x < 1.2
        return 1
    else
        return 0    
    end
end

HS(x) = x < 1 ? Inf : 0

## Prediction of density profiles

In [ ]:
Wall(r;d=3) = r > d ? Inf : 0 #hard walls
Solute(r;R=1) = r < R ? Inf : 0 #hard solute
Vextrad(r) = Wall(r) + Solute(r) #external potential
u = get_params(soft) #pair potential
µ = 0 #chemical potential
T = 1 #temperature
L = 4 #radius of the system
rs,ρ = minimize(L,µ,T,u,Vextrad,get_c1_neural_rad,model;scalewindows = scale_windows) #DFT minimization
plot(rs,ρ,xlabel = "r/σ", ylabel = "ρσ³") #radial density profile, see Fig. 3


In [ ]:
HW(x) = x < 0.5 || x > 4.5 ? Inf : 0 #parallel hard walls
L = 5 #length of periodic rectangular box
u = get_params(ramp) #pair potential
xs, ρ = minimize_plan(L, μ, T, vec(Float32.(u)), HW, xs -> get_c1_neural_plan(model,u)) #DFT minimization
plot(xs,ρ,xlabel = "x/σ", ylabel = "ρσ³") #planar density profile, see Fig. 3

## Pair correlation functions and Ornstein-Zernike inversion

In [ ]:
#homogeneous system
u = get_params(HS)
ρ = 0.34*ones(200) #density profile
xs = collect(0.005:0.01:2-0.005)
c2 = get_c2_from_ρ(ρ,u,xs,model,h) #automatic differentition
allxc2,allc2 = construct_c2(c2;dx=0.01) 
rr,rp,c2mat = construct_c2mat(allxc2,allc2); 

two-body direct correlation functional
$ c_2(r,r';[\rho,\beta\phi]) = \frac{\delta c_1(r;[\rho,\beta\phi])}{\delta \rho(r')}$

In [ ]:
heatmap(rr[1:200], rp[1:200],c2mat[1:200,1:200],clim = (-20,0), aspect_ratio=1, colorbar_title  = "c₂/σ²",xlims=(0, 2),xlabel = "r", ylabel = "r'") # see Fig. 5

symmetric two-body direct correlation functional
$ \bar{c}_2(r,r';[\rho,\beta\phi]) = \frac{c_2(r,r';[\rho,\beta\phi])}{4\pi r'^2}$

In [ ]:
barc2 = c2mat[1:200,1:200]  ./ (4*π* rr[1:200].^2) #symmetric version of c₂
heatmap(rr[1:200], rp[1:200],barc2,clim= (-3.6,0),aspect_ratio=1,colorbar_title  = "barc₂",xlims=(0, 2),xlabel = "r", ylabel = "r'")
# see Fig(6)

radial Ornstein-Zernike equation: <break>
$\bar{h}_2(r,r') = \bar{c}_2(r,r') + 4\pi \int dr'' r''^2 \bar{h}_2(r,r'')\rho(r'')\bar{c}_2(r'',r)$

In [ ]:
#solve Ornstein-Zernike equation
r,ρ2sol = solve_OZ4ρ2(barc2,ρ)
heatmap(r, r,ρ2sol,clim= (0,0.3),aspect_ratio=1,colorbar_title  = "barρ₂σ⁶",xlims=(0, 2),xlabel = "r", ylabel = "r'")

In [ ]:
#inhomogeneous system
u = get_params(soft)
Vextrad(r) = Wall(r) + Solute(r)
µ,T,L = 0,1,4
xs,ρ = minimize(L,µ,T,u,Vextrad,get_c1_neural_rad,model;scalewindows = scale_windows)
plot(xs,ρ,xlabel = "r/σ", ylabel = "ρσ³")# see Fig. 3

In [ ]:
c2 = get_c2_from_ρ(ρ,u,xs,model,h)
allxc2,allc2 = construct_c2(c2;dx=0.01) 
rr,rp,c2mat = construct_c2mat(allxc2,allc2);
barc2 = c2mat[1:400,1:400]  ./ (4*π* xs[1:400].^2)
heatmap(rr[100:300], rp[100:300],barc2[100:300,100:300],clim= (-0.2,0),aspect_ratio=1,colorbar_title  = "barc₂σ²",xlims=(1, 3),xlabel = "r", ylabel = "r'")

In [ ]:
r,ρ2sol = solve_OZ4ρ2(barc2,ρ) 
heatmap(r, r,ρ2sol,clim= (0,0.35),aspect_ratio=1,colorbar_title  = "barρ₂σ⁶",xlims=(1, 3),ylims=(1,3),xlabel = "r", ylabel = "r'")

## Inversion of pair structure

Test particle equation:
$\rho_b g(r) = \exp(c_1(r;[\rho_b g,\beta\phi]) - \beta \phi(r) + \beta \mu)$

In [ ]:
u = get_params(soft)
tp(r) = ϕ(r,u) #external potential = pair potential 
L = 5
rs,ρg = minimize(L,µ,T,u,tp,get_c1_neural_rad,model;scalewindows = scale_windows) #test particle minimization
ρb = sum(ρ[end-100:end])/100 #bulk density
noise = rand(Normal(0, 10e-3) , length(ρg)) ./ (rs .+ 1) .^3; #more or less realistic noise
ρgoal = ρg .+ noise
ρgoal[findall(x -> x < 0, ρgoal)] .= 0
plot(rs,ρgoal,ylim = (0,1.1*maximum(ρg)), xlabel = "r/σ", ylabel = "ρbgσ³")

In [ ]:
dr = 0.01
rs, ϕinv = Inversion(ρgoal, T, ρb, dr)
plot(rs, ϕinv,label="found", xlabel = "r/σ", ylabel = "βϕ")
plot!(rs, u[1:150],label="target",linestyle =:dash)